# Visualizing Presidental Disaster Declarations in Counties Affected by Hurricane Helene

```{image} ../thumbnails/U.S._Census_Bureau_logo_post-2011.svg.png
:alt: US Census Bureau logo
:width: 200px
```

---

## Overview

Hurricane Helene was a devastating Category 4 hurricane that made landfall in Florida's Big Bend on September 26th, 2024. Helene brought massive amounts of rainfall as it moved northward through the Tennessee Valley and the southern Appalachian Mountains (US Army Corps of Engineers - Nashville District, 2025). Storm surge, widespread flooding, extensive power outages, and wind damage were some of the many impacts caused by Hurricane Helene. Presidential disaster declarations were issued for 299 counties across 8 different states, impacting an estimated 26 million people (Sawyer, 2025).

---

## Import Packages

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
from mapclassify import UserDefined
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from metpy.plots import USCOUNTIES

---

## Import CSV Data as a Pandas DataFrame

The dataset is sourced from the US Census Bureau as a part of the [**Census Bureau’s Assessment, Recovery, and Evaluation datasets or CAREs**](https://www.census.gov/programs-surveys/community-resilience-estimates/projects/disaster-recovery/hurricane-helene.html). This dataset curates data from sources such as the Census Bureau, FEMA, DOE, NOAA, and BLS. Data themes include population, exposure, and recovery data for Hurricane Helene.

In [ ]:
helene_data = pd.read_csv('/spare11/atm533/ag265895/bluegreen/data/HELENE.csv') #create pandas dataframe

helene_data

This dataset includes counties that did not issue a presidential disaster declaration during Hurricane Helene, as well as many NaNs or Not a Number values, which we will filter out to clean up the dataset

In [ ]:
helene_data = helene_data.replace([-888888888, -555555555], 0) #replace NaNs with 0

clean_helene_data = helene_data[(helene_data["DR_COUNT"] != 0)] #remove rows with a zero value for major disaster declarations

#calculate and print the number of rows removed 
original_rows = len(helene_data)
print(f"Original number of rows: {original_rows}")

cleaned_rows = len(clean_helene_data)
removed_rows = original_rows - cleaned_rows

print(f"Rows removed: {removed_rows} ")
print(f"Rows remaining: {cleaned_rows}")

#display cleaned up tabular data
clean_helene_data

---

The CAREs dataset does not include the cartographic boundaries of the affected counties, so we are also going to import and merge the [**GeoPandas’ built-in TIGER/Cartographic Boundary files**](https://www2.census.gov/geo/tiger/GENZ2022) from the U.S. Census Bureau to get the polygon geometry of the counties.

In [ ]:
counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_county_5m.zip")

counties

Next, we will drop the columns in the dataset that we don't need.

In [ ]:
counties.drop(columns=['STATEFP', 'COUNTYFP', 'COUNTYNS', 'AFFGEOID', 'STUSPS', 'GEOID', 'LSAD', 'ALAND',	'AWATER'],inplace=True)

counties

Organize the NAMELSAD and STATE_NAME by alphabetical order to match the order from the helene_data columns

In [ ]:
sort_counties = counties.sort_values(by=["STATE_NAME", "NAMELSAD"], ascending=[True, True])

Next, to match the NAME column from the CAREs dataset, we will combine the NAMELSAD column and STATE_NAME column into one column and drop the columns, leaving the geometry column organized in alphabetical order by state and county

In [ ]:
#merge the two columns with a comma separator
sort_counties["NAME"] = sort_counties["NAMELSAD"] + ", " + sort_counties["STATE_NAME"]

sort_counties.drop(columns=['NAMELSAD', 'STATE_NAME'],inplace=True)

sort_counties

Let's make sure all of the words in the NAME column for the sort_location_data and clean_helene_data are properly capitalized in order to ensure a correct merge between datasets

In [ ]:
sort_counties["NAME"] = sort_counties["NAME"].str.title()

clean_helene_data["NAME"] = clean_helene_data["NAME"].str.title()

---

Using the NAME column, we can now merge together the cleane_helene_data columns and sort_location_data

In [ ]:
helene_merged = sort_counties.merge(clean_helene_data, on='NAME', how='inner')

#display the merged dataset
helene_merged

Let's save the new helene_merged dataframe into our data folder so we can use it for future notebooks

In [ ]:
helene_merged.to_csv('../data/helene_merged.csv', index=False)

We can see that the dataset is properly cleaned since 299 counties are left, matching the information from the Overview. Now that the dataset is cleaned up and saved, we can continue to visualize the impacted counties

---

## Visualizing Impacted Counties by Hurricane Helene

First, since we are plotting county border geometries, we need to be sure that the geometries are in the same coordinate reference system (CRS) as the map we are using

In [ ]:
#ensure that helene_merged and counties are in PlateCarree (lon/lat)
helene_merged = helene_merged.to_crs(epsg=4326)
sort_counties = sort_counties.to_crs(epsg=4326)

---

Now, we will visualize all of the counties that issued a presidential disaster declaration for Hurricane Helene. Each county that issued a declaration will be identified in red.

In [ ]:
#plot data
fig = plt.figure(figsize=(12, 10))

#define projection and southeast US
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-95, -75, 24, 39], crs=ccrs.PlateCarree())  

#add geographic features
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.7, edgecolor='black', zorder=2)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), zorder=0)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), zorder=1)

#plot all counties in gray
counties.plot(ax=ax, facecolor='lightgray', edgecolor='gray', linewidth=0.2, transform=ccrs.PlateCarree(), zorder=1)

#plot affected counties in red
helene_merged.plot(ax=ax, facecolor='firebrick', edgecolor='black', linewidth=0.3, transform=ccrs.PlateCarree(), zorder=2)

#title and labels
plt.title("Counties with FEMA Disaster Declarations — Hurricane Helene (2024)", fontsize=14, weight='bold')
plt.show()

We can also visualize the number of Individual Assistance registrations
where the home was destroyed by the disaster using the variable 'destroyed'.

In [ ]:
#plot data
fig = plt.figure(figsize=(12, 10))

#define projection and southeast US
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-95, -75, 24, 39], crs=ccrs.PlateCarree())  

#add geographic features
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.7, edgecolor='grey', zorder=1)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), zorder=0)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), zorder=1)

#add US counties
ax.add_feature(USCOUNTIES.with_scale('20m'), facecolor='none', edgecolor='lightgray', linewidth=0.3, zorder=1)

#plot counties shaded by poverty rate
helene_merged.plot(
    column='destroyed',      
    cmap='Reds',                      
    linewidth=0.2,
    edgecolor='gray',
    legend=True,
    legend_kwds={'label': "# of Registrations", 'shrink': 0.6},
    transform=ccrs.PlateCarree(),
    ax=ax,
    zorder=2
)

#add title
plt.title('Number of Individual Assistance Registrations with Destroyed Homes — Hurricane Helene (2024)', fontsize=14, weight='bold')

plt.show()

How does this compare to the number of valid registrations for the Individual Assistance? We can visualize the relationship between the variables 'destroyed' and 'validReg' with a bar graph.

INSERT HERE ONCE FINISHED

---

## Visualizing Hurricane Helene Impacts in Affected Counties

We can take a deeper look into some of the impacts Hurricane Helene brought that contributed to the presidential disaster declaration by visualizing columns MAX_OUTAGE, MAX_PERCENTAGE_OUT, totPrecip, and Wind_Swath_Max, which look at the maximum amount of customers who experienced a power outage during Helene, the percentage of people who experienced power outages, the amount of percipitation that fell, and the maximum observed wind swath in a county, respectively. 

We'll start with plotting MAX_OUTAGE

In [ ]:
#plot data
fig = plt.figure(figsize=(12, 10))

#define projection and southeast US
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-95, -75, 24, 39], crs=ccrs.PlateCarree())  

#add geographic features
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.7, edgecolor='grey', zorder=1)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), zorder=0)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), zorder=1)

#add US counties
ax.add_feature(USCOUNTIES.with_scale('20m'), facecolor='none', edgecolor='lightgray', linewidth=0.3, zorder=1)

#plot counties shaded by poverty rate
helene_merged.plot(
    column='MAX_OUTAGE',      
    cmap='Oranges',                      
    linewidth=0.2,
    edgecolor='gray',
    legend=True,
    legend_kwds={'label': "# of Power Outages Per Customer", 'shrink': 0.6},
    transform=ccrs.PlateCarree(),
    ax=ax,
    zorder=2
)

#add title
plt.title('Maximum Amount of Customers who Experienced a Power Outage by County — Hurricane Helene (2024)', fontsize=14, weight='bold')

plt.show()

How does this number compare to the number of people living in each county? We can examine the percentage of customers who experienced a power outage by county by using the variable MAX_PERCENTAGE_OUT.

In [ ]:
#plot data
fig = plt.figure(figsize=(12, 10))

#define projection and southeast US
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-95, -75, 24, 39], crs=ccrs.PlateCarree())  

#add geographic features
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.7, edgecolor='grey', zorder=1)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), zorder=0)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), zorder=1)

#add US counties
ax.add_feature(USCOUNTIES.with_scale('20m'), facecolor='none', edgecolor='lightgray', linewidth=0.3, zorder=1)

#plot counties shaded by poverty rate
helene_merged.plot(
    column='MAX_PERCENTAGE_OUT',      
    cmap='Oranges',                      
    linewidth=0.2,
    edgecolor='gray',
    legend=True,
    legend_kwds={'label': "% of Customers Without Power", 'shrink': 0.6},
    transform=ccrs.PlateCarree(),
    ax=ax,
    zorder=2
)

#add title
plt.title('Maximum Percentage of Customers who Experienced a Power Outage by County — Hurricane Helene (2024)', fontsize=14, weight='bold')

plt.show()

The map is very different from the MAX_OUTAGE map as it relates the number of customers without power to the number of customers in the county.

---

When plotting Wind_Swath_Max, the process is different because it has defined amounts for the wind swaths, designating them as tropical storm force, hurricane force, etc. We will use UserDefined to make specific bins.

In [ ]:
#define bins based on dataset
bins = [0, 34, 50, 64]  

#label each bin
bin_labels = [
    'Below Tropical Storm (<34 kt)',
    'Tropical Storm (34–50 kt)',
    '50-Knot Winds (50–64 kt)',
    'Hurricane-Force (64+ kt)'
]

#create classifier
classifier = UserDefined(helene_merged['Wind_Swath_Max'], bins=bins)

Now we can plot Wind_Swath_Max

In [ ]:
#plot data
fig = plt.figure(figsize=(12, 10))

#define projection and southeast US
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-95, -75, 24, 39], crs=ccrs.PlateCarree())  

#add geographic features
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.7, edgecolor='grey', zorder=1)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), zorder=0)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), zorder=1)

#add US counties
ax.add_feature(USCOUNTIES.with_scale('20m'), facecolor='none', edgecolor='lightgray', linewidth=0.3, zorder=1)

# Plot counties with custom bins
plot = helene_merged.plot(
    column='Wind_Swath_Max',
    cmap='Oranges',
    linewidth=0.2,
    edgecolor='gray',
    legend=True,     
    legend_kwds={'title': "Wind Swath (kts)"},
    scheme='UserDefined',
    classification_kwds={'bins': bins},
    transform=ccrs.PlateCarree(),
    ax=ax,
    zorder=2
)

# Add map title
plt.title('Maximum Observed Wind Swath in a County — Hurricane Helene (2024)', fontsize=14, weight='bold')

plt.show()

With this plot, we can see that the strongest wind swaths coincided with the location of Hurricane Helene's landfall. We can follow the same process to plot total precipitation.

While the process is similar to plotting wind swath, we will be defining custom percentile bins to visualize the most significant location of the rainfall totals, totPrecip.

In [ ]:
#define bins based on dataset
p10, p50, p75, p90, p95, p99 = np.percentile(
    helene_merged['totPrecip'].dropna(),
    [10, 50, 75, 90, 95, 99]
)

#define custom percentile bins (in inches)
bins = [p10, p50, p75, p90, p95, p99, helene_merged['totPrecip'].max()]  

#create classifier
classifier = UserDefined(helene_merged['totPrecip'], bins=bins)

In [ ]:
#plot data
fig = plt.figure(figsize=(12, 10))

#define projection and southeast US
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-95, -75, 24, 39], crs=ccrs.PlateCarree())  

#add geographic features
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.7, edgecolor='grey', zorder=1)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), zorder=0)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), zorder=1)

#add US counties
ax.add_feature(USCOUNTIES.with_scale('20m'), facecolor='none', edgecolor='lightgray', linewidth=0.3, zorder=1)

# Plot counties with custom bins
plot = helene_merged.plot(
    column='totPrecip',
    cmap='Blues',
    linewidth=0.2,
    edgecolor='gray',
    legend=True,     
    legend_kwds={'title': "Precipitation Amount (in)"},
    scheme='UserDefined',
    classification_kwds={'bins': bins},
    transform=ccrs.PlateCarree(),
    ax=ax,
    zorder=2
)

# Add map title
plt.title('Total Precipitation by County\n(Percentile-Based Color Encoding) — Hurricane Helene (2024)', fontsize=14, weight='bold')

plt.show()

---

## Summary

This notebook imported two csv datasets and merged them together to retain population, exposure, and recovery data for Hurricane Helene, as well as county geometry to be able to plot the statistical data as polygons. This notebook also spatially visualized the counties that issued a presidential disaster declaration and requested assistance during/after Hurricane Helene in 2024. This notebook also spatially mapped the distribution of customers who lost power during Hurricane Helene, the maximum observed wind swath during Hurricane Helene in each of the affected counties, as well as the total precipitation that fell in inches.